# Querying the announcement text

[abigailhaddad/usajobs-scraping](https://huggingface.co/datasets/abigailhaddad/usajobs-scraping)
is one parquet per month, one row per announcement, 2013-09 to 2026-09. Each row has the
structured API fields *and* the announcement body split by the page's own headings: `jobSummary`,
`majorDuties`, `requirements`, `conditionsOfEmployment`, `qualificationSummary`, `education`,
`additionalInformation`, `benefits`, `howYouWillBeEvaluated`, `requiredDocuments`, `howToApply`,
and the whole-page `text`.

The section columns are the point. The USAJOBS API returns none of them — `MatchedObjectDescriptor`
drops content the announcement page shows — and the Search API only lists jobs open right now, so
anything that opened and closed between two collection runs never existed as far as the API is
concerned.

Four questions below. Three have useful answers. The fourth is a query that looks great and means
nothing, which is the one I'd read if you only read one.

## Setup

DuckDB reads the parquet over HTTP and fetches only the columns a query touches, so nothing here
needs a download. Two things matter in practice:

**Use a token.** Anonymous reads get HTTP 429 partway through anything non-trivial. A read token
from https://huggingface.co/settings/tokens is enough.

**Don't touch `text`.** It is the whole rendered page and dwarfs the rest. Querying the eleven
section columns is what keeps these to seconds; adding `text` to any of them trips the rate limit.

In [1]:
import os
import duckdb

con = duckdb.connect()
con.execute("SET threads=2")

token = os.environ.get("HF_TOKEN")
if token:
    con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{token}')")
    print("authenticated")
else:
    print("no HF_TOKEN - expect HTTP 429 on all but the smallest query")

# Two months as a sample. See the last section for widening this.
MONTHS = ("'hf://datasets/abigailhaddad/usajobs-scraping/data/2025_03.parquet', "
          "'hf://datasets/abigailhaddad/usajobs-scraping/data/2025_06.parquet'")
con.execute(f"CREATE VIEW jobs AS SELECT * FROM read_parquet([{MONTHS}])")

# Both halves of every mention, for the degree section below.
con.execute(r'''
CREATE VIEW degree_windows AS
SELECT usajobsControlNumber, positionTitle, occupationalSeries,
       regexp_extract_all(
         lower(coalesce(education,'') || ' ~~ ' || coalesce(qualificationSummary,'')),
         '.{0,250}(ph\.?\s?d|doctoral|doctorate).{0,250}') AS mentions
FROM jobs
''')

con.sql("SELECT count(*) AS announcements FROM jobs").show()

authenticated


┌───────────────┐
│ announcements │
│     int64     │
├───────────────┤
│         20697 │
└───────────────┘



## 1. Who needs a doctorate

Degree requirements live in `education` and `qualificationSummary`. No structured field carries
this, in either API, so the question had no answer before.

Start with the obvious query.

In [2]:
con.sql(r'''
SELECT count(*) AS announcements,
       count(*) FILTER (
         WHERE regexp_matches(lower(coalesce(education,'') || ' ' || coalesce(qualificationSummary,'')),
                              'ph\.?\s?d|doctoral|doctorate')) AS mentions_a_doctorate
FROM jobs
''').show()

┌───────────────┬──────────────────────┐
│ announcements │ mentions_a_doctorate │
│     int64     │        int64         │
├───────────────┼──────────────────────┤
│         20697 │                 4623 │
└───────────────┴──────────────────────┘



22%. That is not 22% of federal jobs requiring a doctorate, and the gap is about two orders of
magnitude. Read what the matches say.

In [3]:
con.sql('''
SELECT DISTINCT substr(mentions[1], 1, 210) AS a_mention
FROM degree_windows
WHERE len(mentions) > 0
ORDER BY 1
LIMIT 3
''').show(max_width=180)

┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                                                    a_mention                                                                                     │
│                                                                                     varchar                                                                                      │
├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│  (4) colloabortating with other law enforcement agencies, management and specialists to implement resource protection and law enforcement. -or- education: successful completion │
│  of 3 years of progressively highe                                                           

Two of those three offer a doctorate as an alternative rather than requiring it — note `-or-
education:` in the first and "experience or education: candidates at this level must meet one of
the following" in the third. That is how General Schedule qualification standards are written: a
degree is one route in, experience is another, and the announcement lists both.

So "Ph.D." appearing in an announcement tells you nothing about whether the job needs one.
Classifying on the framing around each mention starts to separate the two uses.

In [4]:
con.sql(r'''
SELECT count(*) FILTER (WHERE len(mentions) > 0) AS mentions,
       count(*) FILTER (WHERE list_bool_or(list_transform(mentions, lambda m:
         regexp_matches(m, 'in lieu of|substitut|combination of education and experience'
                           '|graduate education leading to such a degree')))) AS substitution_framing,
       count(*) FILTER (WHERE list_bool_or(list_transform(mentions, lambda m:
         regexp_matches(m, 'basic requirement|individual occupational requirement'
                           '|must (have|possess|hold) a')))) AS requirement_framing
FROM degree_windows
''').show()

┌──────────┬──────────────────────┬─────────────────────┐
│ mentions │ substitution_framing │ requirement_framing │
│  int64   │        int64         │        int64        │
├──────────┼──────────────────────┼─────────────────────┤
│     4623 │                 1647 │                1178 │
└──────────┴──────────────────────┴─────────────────────┘



The requirement-framed set concentrates hard, which is the first sign it is picking up something
real.

In [5]:
con.sql(r'''
SELECT occupationalSeries, any_value(positionTitle) AS example, count(*) AS n
FROM degree_windows
WHERE list_bool_or(list_transform(mentions, lambda m:
        regexp_matches(m, 'basic requirement|individual occupational requirement'
                          '|must (have|possess|hold) a')))
  AND NOT list_bool_or(list_transform(mentions, lambda m:
        regexp_matches(m, 'in lieu of|substitut|combination of education and experience'
                          '|graduate education leading to such a degree')))
GROUP BY occupationalSeries
ORDER BY n DESC
LIMIT 8
''').show(max_width=100)

┌────────────────────┬─────────────────────────────────────────────────────────────────────┬───────┐
│ occupationalSeries │                               example                               │   n   │
│      varchar       │                               varchar                               │ int64 │
├────────────────────┼─────────────────────────────────────────────────────────────────────┼───────┤
│ 0180               │ Psychologist (PTSD EBP) - Recruitment/Relocation Incentive & Educa… │   300 │
│ 0610               │ Nurse Practitioner - Mental Health                                  │   195 │
│ 0183               │ LICENSED PROF MENTAL HEALTH COUNSELOR-MENTAL HEALTH REHABILITATION… │   119 │
│ 0665               │ Audiologist (Clinical Specialist)                                   │    54 │
│ 0182               │ MARRIAGE & FAMILY THERAPIST-MENTAL HEALTH RESIDENTIAL REHABILITATI… │    28 │
│ 1224               │ Patent Examiner (Electrical Engineering)                            

Psychology, nurse practitioners, mental health counselling, audiology, pharmacy. Plausible.

It is still wrong, though, and you can only tell by reading. Compare the top two series:

In [6]:
for series in ("0180", "0610"):
    row = con.sql(rf'''
        SELECT list_filter(mentions, lambda m:
                 regexp_matches(m, 'doctoral degree|ph\.?\s?d'))[1][1:400]
        FROM degree_windows
        WHERE occupationalSeries = '{series}' AND len(mentions) > 0
        ORDER BY usajobsControlNumber LIMIT 1
    ''').fetchone()
    print(f"===== series {series} =====\n  {row[0]}\n")

===== series 0180 =====
  sic requirements : a. united states citizenship: be a citizen of the united states. non-citizens may be appointed when it is not possible to recruit qualified citizens in accordance with chapter 3, section a, paragraph 3.g,. b. education: (1) have a doctoral degree in psychology from a graduate program in psychology accredited by the american psychological association (apa), the psychological clin



===== series 0610 =====
  ing a wide range of direct patient care services, performing charge nurse/team lead, orienting new employees, experience in providing data reports and quality assessments and performance improvement projects; or education: successful completion of a phd or equivalent doctoral degree from a professional nursing educational program or related medical science field. time in grade federal employees in



0180 requires one: "education: (1) have a doctoral degree in psychology from a graduate program
accredited by the American Psychological Association."

0610 does not. Its text reads "... **or** education: successful completion of a PhD or equivalent
doctoral degree from a professional nursing educational program" — the doctorate is an alternative
to the experience listed before it. Same for the patent examiners at 1224, whose requirement is a
lettered list where the PhD is option (a).

So the honest answer for this sample sits between about 15 announcements, if you demand phrasing as
explicit as "have a doctoral degree", and 1,178, which over-counts. The occupational concentration
is real; the exact count is a reading-comprehension problem, and the right tool for the last mile
is an LLM pass over the 4,623 windows rather than a longer regex.

What changed is that the question is now answerable at all. None of this text exists in the API.

## 2. Where the structured field and the prose disagree

Each row carries both halves: `securityClearanceRequired` from the API, and whatever the
announcement itself says about clearances. Only the second one binds an applicant.

In [7]:
con.sql('''
SELECT count(*) FILTER (WHERE securityClearanceRequired = 'N') AS field_says_none,
       count(*) FILTER (WHERE securityClearanceRequired = 'N'
         AND lower(coalesce(requirements,'') || coalesce(conditionsOfEmployment,''))
             LIKE '%security clearance%') AS but_the_text_mentions_one
FROM jobs
''').show()

┌─────────────────┬───────────────────────────┐
│ field_says_none │ but_the_text_mentions_one │
│      int64      │           int64           │
├─────────────────┼───────────────────────────┤
│            9883 │                       299 │
└─────────────────┴───────────────────────────┘



In [8]:
con.sql(r'''
SELECT regexp_extract(lower(coalesce(requirements,'') || ' | ' || coalesce(conditionsOfEmployment,'')),
                      '.{0,45}security clearance.{0,120}', 0) AS context
FROM jobs
WHERE securityClearanceRequired = 'N'
  AND lower(coalesce(requirements,'') || coalesce(conditionsOfEmployment,'')) LIKE '%security clearance%'
ORDER BY usajobsControlNumber
LIMIT 3
''').show(max_width=180)

┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                                                     context                                                                                      │
│                                                                                     varchar                                                                                      │
├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│  or be able to acquire the appropriate level security clearance for this position. qualifications military grades: enl: pv1 to sgt *all applications will be considered regardle │
│ ss of b                                                                                      

"Must possess or be able to acquire the appropriate level security clearance for this position," on
a posting whose clearance field is `N`. About 3% of the no-clearance postings say something like
it. I checked the sample for the obvious false positive — a posting saying no clearance is needed —
and did not find one.

Whether the field is wrong or is answering a narrower question than it looks like, somebody
filtering jobs on it is being misled, and the prose is the only way to know.

## 3. Direct hire authority

Direct hire authority lets an agency skip competitive rating and veterans' preference. Whether a
posting uses one is stated in the announcement and captured by no field in either API.

In [9]:
con.sql('''
SELECT hiringDepartmentName AS department, count(*) AS direct_hire_postings
FROM jobs
WHERE lower(coalesce(additionalInformation,'') || coalesce(requirements,'')) LIKE '%direct hire%'
GROUP BY department
ORDER BY direct_hire_postings DESC
LIMIT 6
''').show(max_width=100)

┌────────────────────────────────┬──────────────────────┐
│           department           │ direct_hire_postings │
│            varchar             │        int64         │
├────────────────────────────────┼──────────────────────┤
│ Department of the Navy         │                   87 │
│ Department of Defense          │                   78 │
│ Department of the Army         │                   26 │
│ Department of the Interior     │                   25 │
│ Department of Veterans Affairs │                    9 │
│ Department of Transportation   │                    4 │
└────────────────────────────────┴──────────────────────┘



246 of 20,697, about 1.2%, and overwhelmingly Defense. The announcements name the specific
authority, which is the part worth having:

> "this announcement uses the **defense industrial base, major range and test facilities direct
> hire authority** to recruit and appoint qualified candidates to certain positions in the
> competitive service"

One warning from checking this one. Screening out negations with `(not|non|no longer).{0,40}direct
hire` returns six hits, and all six are genuine direct-hire postings: `not` matches inside "public
**not**ice", and the rest say "veterans preference does not apply to announcements posted under
direct hire authority", which is a direct-hire posting describing itself. The negation check was
the buggy part, not the original query.

## 4. A query that looks great and means nothing

`requiredDocuments` lists what you have to submit, so:

In [10]:
con.sql('''
SELECT count(*) AS announcements,
       count(*) FILTER (WHERE lower(requiredDocuments) LIKE '%transcript%') AS transcript,
       count(*) FILTER (WHERE lower(requiredDocuments) LIKE '%sf-50%')      AS sf_50,
       count(*) FILTER (WHERE lower(requiredDocuments) LIKE '%dd-214%')     AS dd_214
FROM jobs
''').show()

┌───────────────┬────────────┬───────┬────────┐
│ announcements │ transcript │ sf_50 │ dd_214 │
│     int64     │   int64    │ int64 │ int64  │
├───────────────┼────────────┼───────┼────────┤
│         20697 │      15660 │ 16501 │  12122 │
└───────────────┴────────────┴───────┴────────┘



80% of federal postings require an SF-50 and 59% require a DD-214? No. Read them.

In [11]:
for doc in ("sf-50", "dd-214"):
    rows = con.sql(f'''
        SELECT regexp_extract(lower(requiredDocuments), '.{{0,95}}{doc}.{{0,110}}', 0)
        FROM jobs WHERE lower(requiredDocuments) LIKE '%{doc}%'
        ORDER BY usajobsControlNumber LIMIT 2
    ''').fetchall()
    print(f"===== {doc} =====")
    for (r,) in rows:
        print(" ", r, "\n")

===== sf-50 =====
  -transition/ctap_guideline.pdf . current or former federal employee : include your most recent sf-50 or if reinstatement eligible include your career sf-50. noncompetitive eligibles : submit additional document 

  ral civil service employee, you must also submit a copy of a notification of personnel action (sf-50); previously, federal employees must also submit: sf-50 showing your current or former civil service status;  



===== dd-214 =====
  ns' preference or claiming sole survivorship preference? you must submit a copy of your latest dd-214 certificate of release or discharge from active duty (any copy that shows all dates of service, as well as ch 

  ence, you must upload documentation to support your veteran's preference claim to include your dd-214 indicating the type of discharge. if claiming veteran's preference for a disability, you must submit a va dis 



Conditional, every time. The SF-50 is for people who already work or used to work for the federal
government. The DD-214 is for people claiming veterans' preference. Neither is a requirement on
applicants in general — they are standard paragraphs addressed to whichever applicants they apply
to, and they appear on nearly every announcement. The counts are correct and the interpretation is
worthless: they measure use of the boilerplate.

Transcripts are the genuinely interesting one, because both readings exist in the data. Some
postings put a transcript in the mandatory list:

> "to apply for this position, you must provide a complete application package which includes:
> license professional certification resume **transcript**"

and others make it conditional:

> "**if you meet this requirement based on education you must submit** a copy of your transcript"

Telling those apart is the actual question, and a `LIKE '%transcript%'` count is not an
approximation of it.

## Two DuckDB traps, since I hit both writing this

`SIMILAR TO` is a full-string regex match and `%` is not a wildcard in it, so
`x SIMILAR TO '%(phd)%'` is always false. Use `LIKE '%phd%'` or `regexp_matches(x, 'phd')`.

`USING SAMPLE n ROWS` is applied to the table scan, before the `WHERE`. On a filtered query it will
happily return zero rows. Sample inside a subquery, or use `ORDER BY ... LIMIT n` as everything
above does.

## Scaling up

```python
con.execute("CREATE VIEW jobs AS SELECT * FROM "
            "read_parquet('hf://datasets/abigailhaddad/usajobs-scraping/data/2025_*.parquet')")
```

A year at a time, with a token. All 152 files in one glob gets rate limited regardless.

One thing not to use: the filter and search on the dataset's HuggingFace page. HuggingFace indexes
a fixed slice of a dataset this size, and here that slice is 99,059 rows out of 3,229,043, about
3%, with nothing on the page saying so. Counts taken from it are a silent sample. Read the
parquet.